# Build Pipeline Model
This notebook merges cleaned contacts and deals data, generates probability scores, and calculates weighted pipeline value for revenue forecasting.


In [52]:
import pandas as pd

# Load cleaned datasets
contacts = pd.read_csv("../data_clean/contacts_clean.csv")
deals = pd.read_csv("../data_clean/deals_clean.csv")

contacts.head(), deals.head()



(   contact_id first_name last_name                            email  \
 0           1     Amanda     Lopez  amanda.lopez@northbridgeops.com   
 1           2     Marcus      Chen          marcus.chen@revpoint.ca   
 2           3      Sarah     Patel         sarah.patel@clearwave.io   
 3           4     Daniel      Reed        daniel.reed@skylinehr.com   
 4           5      Emily    Nguyen       emily.nguyen@brightpath.ca   
 
                company           job_title lifecycle_stage created_date  \
 0      Northbridge Ops  Operations Manager            Lead   2024-01-14   
 1   RevPoint Solutions   Director of Sales             MQL   2024-02-03   
 2  ClearWave Analytics   Marketing Manager            Lead   2024-03-22   
 3           Skyline HR                 COO             SQL   2024-04-10   
 4   BrightPath Systems      RevOps Manager            Lead   2024-05-18   
 
           owner  
 0     Jamie Lee  
 1     Jamie Lee  
 2   Alex Morgan  
 3   Alex Morgan  
 4  Jordan Sm

import pandas as pd

contacts = pd.read_csv("../data_clean/contacts_clean.csv")
deals = pd.read_csv("../data_clean/deals_clean.csv")

contacts.head(), deals.head()


In [53]:
pipeline = pd.merge(
    deals,
    contacts,
    how="left",
    on="contact_id"
)

pipeline.head()



,deal_id,contact_id,deal_name,deal_stage,amount,created_date_x,close_date,owner_x,first_name,last_name,email,company,job_title,lifecycle_stage,created_date_y,owner_y
0,1,12,Enterprise RevOps Suite,Prospecting,45000,2024-01-10,NaN,Alex Morgan,Kevin,Brooks,kevin.brooks@apexlogic.io,ApexLogic,CTO,SQL,2024-01-08,Alex Morgan
1,2,3,Analytics Dashboard Upgrade,Qualification,18000,2024-01-12,2024-02-15,Alex Morgan,Sarah,Patel,sarah.patel@clearwave.io,ClearWave Analytics,Marketing Manager,Lead,2024-03-22,Alex Morgan
2,3,7,CRM Integration Package,Proposal,22000,2024-01-14,NaN,Jamie Lee,Olivia,Grant,olivia.grant@mapletech.ca,MapleTech,Sales Manager,Lead,2024-06-27,Jamie Lee
3,4,1,Workflow Automation Build,Closed Won,32000,2024-01-16,2024-02-02,Jamie Lee,Amanda,Lopez,amanda.lopez@northbridgeops.com,Northbridge Ops,Operations Manager,Lead,2024-01-14,Jamie Lee
4,5,5,Sales Pipeline Optimization,Negotiation,27000,2024-01-18,NaN,Jordan Smith,Emily,Nguyen,emily.nguyen@brightpath.ca,BrightPath Systems,RevOps Manager,Lead,2024-05-18,Jordan Smith


pipeline = pd.merge(
    deals,
    contacts,
    how="left",
    on="contact_id"
)

pipeline.head()


In [54]:
stage_probabilities = {
    "Prospecting": 0.10,
    "Qualified": 0.25,
    "Proposal": 0.50,
    "Negotiation": 0.75,
    "Closed Won": 1.00,
    "Closed Lost": 0.00
}

pipeline["probability"] = pipeline["deal_stage"].map(stage_probabilities).fillna(0)
pipeline[["deal_stage", "probability"]].head()



,deal_stage,probability
0,Prospecting,0.10
1,Qualification,0.00
2,Proposal,0.50
3,Closed Won,1.00
4,Negotiation,0.75


stage_probabilities = {
    "Prospecting": 0.10,
    "Qualified": 0.25,
    "Proposal": 0.50,
    "Negotiation": 0.75,
    "Closed Won": 1.00,
    "Closed Lost": 0.00
}

pipeline["probability"] = pipeline["deal_stage"].map(stage_probabilities).fillna(0)
pipeline[["deal_stage", "probability"]].head()


In [55]:
pipeline["deal_value"] = pipeline["amount"].fillna(0)
pipeline["weighted_value"] = pipeline["deal_value"] * pipeline["probability"]

pipeline[["deal_name", "deal_stage", "deal_value", "probability", "weighted_value"]].head()



,deal_name,deal_stage,deal_value,probability,weighted_value
0,Enterprise RevOps Suite,Prospecting,45000,0.10,4500.0
1,Analytics Dashboard Upgrade,Qualification,18000,0.00,0.0
2,CRM Integration Package,Proposal,22000,0.50,11000.0
3,Workflow Automation Build,Closed Won,32000,1.00,32000.0
4,Sales Pipeline Optimization,Negotiation,27000,0.75,20250.0


pipeline["deal_value"] = pipeline["amount"].fillna(0)
pipeline["weighted_value"] = pipeline["deal_value"] * pipeline["probability"]

pipeline[["deal_name", "deal_stage", "deal_value", "probability", "weighted_value"]].head()


In [56]:
summary_by_stage = (
    pipeline.groupby("deal_stage")[["deal_value", "weighted_value"]]
    .sum()
    .reset_index()
)

summary_by_stage



,deal_stage,deal_value,weighted_value
0,Closed Lost,742000,0.0
1,Closed Won,2098000,2098000.0
2,Negotiation,3260000,2445000.0
3,Proposal,2630000,1315000.0
4,Prospecting,2187000,218700.0
5,Qualification,2251000,0.0


import seaborn as sns
import matplotlib.pyplot as plt

sns.barplot(data=pipeline, x="deal_stage", y="weighted_value", estimator=sum)
plt.xticks(rotation=45)
plt.title("Weighted Pipeline Value by Stage")
plt.show()


In [57]:
if "owner" in pipeline.columns:
    summary_by_owner = (
        pipeline.groupby("owner")[["deal_value", "weighted_value"]]
        .sum()
        .reset_index()
    )
    summary_by_owner



pipeline.to_csv("../data_clean/pipeline_model.csv", index=False)


In [58]:
pipeline.to_csv("../data_clean/pipeline_model.csv", index=False)
summary_by_stage.to_csv("../data_clean/pipeline_summary_by_stage.csv", index=False)

if "owner" in pipeline.columns:
    summary_by_owner.to_csv("../data_clean/pipeline_summary_by_owner.csv", index=False)



##  Pipeline Model Complete
The dataset is now ready for dashboarding and further analysis.
